# Pathway Enrichment — Cross-Ancestry, Cross-Phenotype Gene Set Comparison

**Purpose:** Test whether AA and EA gene sets (from the fully corrected causal
networks) converge on shared biological pathways despite near-zero SNP-level
overlap. Uses Enrichr (public API, no local install needed).

**Method:** For each of the 4 gene sets (AA smoking, AA CPD, EA smoking, EA CPD),
submit only protein-coding/named genes (excluding intergenic and unnamed hits)
to Enrichr, query against GO Biological Process, KEGG, and Reactome libraries.

**Important honest caveat, stated upfront:** these gene lists are small (3-8
genes each). Enrichment tests have limited statistical power at this scale —
results here are exploratory/suggestive, not confirmatory. Report accordingly.

In [2]:
import requests
import pandas as pd
import time
import os

OUT_DIR = r"C:\Users\user\Desktop\ai causal\causal_project\cross_ancestry\enrichment"
os.makedirs(OUT_DIR, exist_ok=True)

gene_sets = {
    "AA_smoking": ["PPP1R12B", "CELA1", "LAMA3", "CASC8", "FAM126A", "ZNF79", "FLG-AS1", "PRSS54"],
    "AA_CPD": ["PPP1R12B", "HEMK1"],
    "EA_smoking": ["HUS1", "CPT1A", "ADAMTS13", "POLI", "SPATA1"],
    "EA_CPD": ["AHRR", "ARFGAP3", "EMILIN2", "PSG10P"],
}

def run_enrichr(genes, description):
    add_url = "https://maayanlab.cloud/Enrichr/addList"
    genes_str = "\n".join(genes)
    payload = {"list": (None, genes_str), "description": (None, description)}
    resp = requests.post(add_url, files=payload)
    resp.raise_for_status()
    data = resp.json()
    return data["userListId"]

def get_enrichment(user_list_id, library):
    enrich_url = "https://maayanlab.cloud/Enrichr/enrich"
    params = {"userListId": user_list_id, "backgroundType": library}
    resp = requests.get(enrich_url, params=params)
    resp.raise_for_status()
    return resp.json()

libraries = ["GO_Biological_Process_2023", "KEGG_2021_Human", "Reactome_2022"]

all_results = {}
for set_name, genes in gene_sets.items():
    print(f"\n=== {set_name} ({len(genes)} genes) ===")
    try:
        list_id = run_enrichr(genes, set_name)
        time.sleep(1)
        for lib in libraries:
            result = get_enrichment(list_id, lib)
            terms = result.get(lib, [])[:10]
            print(f"\n  {lib} (top hits):")
            for term in terms:
                rank, name, pval, zscore, combined_score, overlap_genes = term[0], term[1], term[2], term[3], term[4], term[5]
                adj_pval = term[6] if len(term) > 6 else None
                adj_pval_str = f"{adj_pval:.4g}" if adj_pval is not None else "N/A"
                print(f"    {name} | p={pval:.4g} | adj_p={adj_pval_str} | genes: {overlap_genes}")
            all_results[f"{set_name}_{lib}"] = terms
            time.sleep(1)
    except Exception as e:
        print(f"  Error for {set_name}: {e}")

print("\n\nDone.")


=== AA_smoking (8 genes) ===

  GO_Biological_Process_2023 (top hits):
    Regulation Of Muscle System Process (GO:0090257) | p=0.004392 | adj_p=0.02505 | genes: ['PPP1R12B']
    Proteolysis (GO:0006508) | p=0.007117 | adj_p=0.02505 | genes: ['CELA1', 'PRSS54']
    Regulation Of Muscle Contraction (GO:0006937) | p=0.01154 | adj_p=0.02505 | genes: ['PPP1R12B']
    Endodermal Cell Differentiation (GO:0035987) | p=0.01233 | adj_p=0.02505 | genes: ['LAMA3']
    Endoderm Formation (GO:0001706) | p=0.01392 | adj_p=0.02505 | genes: ['LAMA3']
    Epidermis Development (GO:0008544) | p=0.0335 | adj_p=0.05026 | genes: ['LAMA3']
    Negative Regulation Of Transcription By RNA Polymerase II (GO:0000122) | p=0.2675 | adj_p=0.3439 | genes: ['ZNF79']
    Negative Regulation Of DNA-templated Transcription (GO:0045892) | p=0.3436 | adj_p=0.3865 | genes: ['ZNF79']
    Regulation Of Transcription By RNA Polymerase II (GO:0006357) | p=0.5749 | adj_p=0.5749 | genes: ['ZNF79']

  KEGG_2021_Human (top hits)